# PokeAPI + Ollama
Chatbot de Pokémon com sando um modelo local via Ollama.

## Instalação

In [ ]:
!uv pip install pokebase openai rich

Using Python 3.12.13 environment at: /usr
Checked 3 packages in 255ms


## Iniciar Ollama e baixar modelo

In [ ]:
# Instala o Ollama
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2build1).
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, threading, time

def run_ollama():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

thread = threading.Thread(target=run_ollama, daemon=True)
thread.start()
time.sleep(2)
print("Ollama rodando em http://localhost:11434")

Ollama rodando em http://localhost:11434


In [ ]:
# Modelos:
#   llama3.1:8b   → boa relação tamanho/qualidade
#   qwen2.5:7b    → excelente em tool use
#   mistral-nemo  → também sólido

MODEL = "llama3.1:8b"

!ollama pull {MODEL}

## Funções da PokéAPI

In [ ]:
import pokebase as pb
from functools import lru_cache

ALL_TYPES = [
    "normal", "fire", "water", "electric", "grass", "ice",
    "fighting", "poison", "ground", "flying", "psychic",
    "bug", "rock", "ghost", "dragon", "dark", "steel", "fairy"
]


@lru_cache(maxsize=256)
def _get_pokemon(nome: str):
    try:
        return pb.pokemon(nome.lower())
    except Exception:
        return None


@lru_cache(maxsize=256)
def _get_type(tipo: str):
    try:
        return pb.type_(tipo.lower())
    except Exception:
        return None


def get_pokemon_types(pokemon_name: str) -> dict:
    poke = _get_pokemon(pokemon_name)
    if not poke:
        return {"error": f"Pokémon '{pokemon_name}' não encontrado."}
    return {"pokemon_name": pokemon_name, "types": [t.type.name for t in poke.types]}


def get_pokemons_of_type(pokemon_type: str) -> dict:
    tipo = _get_type(pokemon_type)
    if not tipo:
        return {"error": f"Tipo '{pokemon_type}' não encontrado."}
    return {"type": pokemon_type, "pokemons": [p.pokemon.name for p in tipo.pokemon]}


def get_type_defense(pokemon_type: str) -> dict:
    tipo = _get_type(pokemon_type)
    if not tipo:
        return {"error": f"Tipo '{pokemon_type}' não encontrado."}
    relations = tipo.damage_relations
    return {
        "type": pokemon_type,
        "resistencias": [t.name for t in relations.half_damage_from],
        "fraquezas": [t.name for t in relations.double_damage_from],
        "imunidades": [t.name for t in relations.no_damage_from],
    }


def get_pokemon_defense(pokemon_name: str) -> dict:
    poke = _get_pokemon(pokemon_name)
    if not poke:
        return {"error": f"Pokémon '{pokemon_name}' não encontrado."}

    poke_types = [t.type.name for t in poke.types]
    defense = {t: 1.0 for t in ALL_TYPES}

    for p_type in poke_types:
        tipo = _get_type(p_type)
        if not tipo:
            continue
        relations = tipo.damage_relations
        for t in relations.half_damage_from:
            defense[t.name] *= 0.5
        for t in relations.double_damage_from:
            defense[t.name] *= 2.0
        for t in relations.no_damage_from:
            defense[t.name] = 0.0

    resultado = {"pokemon_name": pokemon_name, "types": poke_types,
                 "muito_fraco": [], "fraco": [], "neutro": [],
                 "resistente": [], "muito_resistente": [], "imune": []}

    for tipo, mult in defense.items():
        if mult == 0:      resultado["imune"].append(tipo)
        elif mult == 0.25: resultado["muito_resistente"].append(tipo)
        elif mult == 0.5:  resultado["resistente"].append(tipo)
        elif mult == 2:    resultado["fraco"].append(tipo)
        elif mult == 4:    resultado["muito_fraco"].append(tipo)
        else:              resultado["neutro"].append(tipo)

    return resultado


def get_pokemon_stats(pokemon_name: str) -> dict:
    poke = _get_pokemon(pokemon_name)
    if not poke:
        return {"error": f"Pokémon '{pokemon_name}' não encontrado."}
    stats = {s.stat.name: s.base_stat for s in poke.stats}
    return {"pokemon_name": pokemon_name, "stats": stats, "total": sum(stats.values())}


# Mapeamento nome → função
TOOLS_MAP = {
    "get_pokemon_types": get_pokemon_types,
    "get_pokemons_of_type": get_pokemons_of_type,
    "get_type_defense": get_type_defense,
    "get_pokemon_defense": get_pokemon_defense,
    "get_pokemon_stats": get_pokemon_stats,
}

print("Funções carregadas!")

Funções carregadas!


## Configuração do cliente Ollama

In [ ]:
import json
from openai import OpenAI
from rich import print as rprint
from rich.panel import Panel

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

# Definição das tools no formato OpenAI
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_pokemon_types",
            "description": "Retorna os tipos de um Pokémon (ex: Fire, Flying para Charizard).",
            "parameters": {
                "type": "object",
                "properties": {
                    "pokemon_name": {"type": "string", "description": "Nome do Pokémon em minúsculas."}
                },
                "required": ["pokemon_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_pokemons_of_type",
            "description": "Lista todos os Pokémon de um determinado tipo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pokemon_type": {"type": "string", "description": "Nome do tipo em inglês (ex: fire, water, dragon)."}
                },
                "required": ["pokemon_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_type_defense",
            "description": "Retorna fraquezas, resistências e imunidades de um tipo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pokemon_type": {"type": "string", "description": "Nome do tipo (ex: fire, psychic)."}
                },
                "required": ["pokemon_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_pokemon_defense",
            "description": "Retorna a tabela defensiva completa de um Pokémon com multiplicadores reais (4x, 2x, 1x, 0.5x, 0.25x, 0x).",
            "parameters": {
                "type": "object",
                "properties": {
                    "pokemon_name": {"type": "string", "description": "Nome do Pokémon."}
                },
                "required": ["pokemon_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_pokemon_stats",
            "description": "Retorna os stats base de um Pokémon: HP, Ataque, Defesa, Sp. Atk, Sp. Def, Speed e total.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pokemon_name": {"type": "string", "description": "Nome do Pokémon."}
                },
                "required": ["pokemon_name"]
            }
        }
    }
]

SYSTEM_PROMPT = (
    "Você é um especialista em Pokémon. "
    "Sempre use as tools disponíveis para buscar dados reais antes de responder — nunca invente informações. "
    "Responda em português, de forma clara e amigável."
)

print(f"Cliente configurado para {MODEL}")

Cliente configurado para llama3.1:8b


## Loop de conversa com tool use

In [ ]:
def executar_tool(tool_name: str, tool_input: dict) -> str:
    if tool_name not in TOOLS_MAP:
        return json.dumps({"error": f"Tool '{tool_name}' não encontrada."})
    try:
        return json.dumps(TOOLS_MAP[tool_name](**tool_input), ensure_ascii=False)
    except Exception as e:
        return json.dumps({"error": str(e)})


def chat_com_pokemon(pergunta: str, messages: list = None) -> tuple[str, list]:
    if messages is None:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    messages.append({"role": "user", "content": pergunta})

    # Loop agentic: continua enquanto o modelo quiser usar tools
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            return msg.content or "(sem resposta)", messages

        # Executa cada tool chamada e devolve os resultados
        for tool_call in msg.tool_calls:
            nome = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            rprint(f"[dim] Chamando: [bold]{nome}[/bold]({args})[/dim]")

            resultado = executar_tool(nome, args)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": resultado,
            })


print("Funções de chat prontas!")

Funções de chat prontas!


## Pergunta única

In [ ]:
pergunta = input("Faça sua pergunta sobre Pokémon: ")

resposta, _ = chat_com_pokemon(pergunta)
rprint(Panel(resposta, title="Ollama", border_style="green"))

## Modo conversa (multi-turn com memória)

In [ ]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat Pokémon iniciado! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Você: ").strip()
    if pergunta.lower() in ("sair", "exit", "quit"):
        print("Até mais!")
        break
    if not pergunta:
        continue

    resposta, historico = chat_com_pokemon(pergunta, historico)
    rprint(f"\n[bold green]Ollama:[/bold green] {resposta}\n")

Chat Pokémon iniciado! Digite 'sair' para encerrar.



 Chamando: get_pokemons_of_type({'pokemon_type': 'fire'})

Ollama: Pokémon do mesmo tipo que charmander são: Vulpix, Ninetales, Charmeleon, Charizard, Magmar, Flareon, 
Moltres, Cyndaquil, Quilava, Typhlosion, Slugma, Magcargo, Houndour, Houndoom, Numel, Camerupt, Torkoal, Chimchar, 
Monferno, Infernape, Magmortar, Heatran, Victini, Tepig, Pignite, Emboar, Pansear, Simisear, Fennekin, Braixen, 
Delphox, Fletchinder, Talonflame, Litleo, Pyroar, Volcanion.